# ISHate — Ocampo et al. (EACL 2023) Replication

Evaluates BERT, HateBERT, RoBERTa (baseline and RAC) on the two 3-class tasks from Ocampo et al.

## 1. Imports

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
from sklearn.metrics import (
    classification_report,
    precision_recall_fscore_support,
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))
from retriever import retrieve_top_k_above_threshold

## 2. Configuration

In [ ]:
# Define paths
ROOT_DIR        = Path('../..')
WEIGHTS_DIR     = ROOT_DIR / 'weigths' / 'weights_baseline'
WEIGHTS_RAC_DIR = ROOT_DIR / 'weigths' / 'weights_rac_best_hyperparameters'
INDEX_DIR       = ROOT_DIR / 'corpus' / 'index'

# Model config
MAX_LENGTH = 256
BATCH_SIZE = 32
NUM_EPOCHS = 3
LR         = 2e-5

# Retrieval config
K           = 5
THRESHOLD   = 0.3
SBERT_HF_ID = 'sentence-transformers/all-mpnet-base-v2'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 3. Load ISHate

Load train/val/test splits with 3-class labels for Tasks A and B.

In [ ]:
from data_loaders import load_ishate_multiclass

TASK_A_NAMES = ['Non-HS', 'Explicit HS', 'Implicit HS']
TASK_B_NAMES = ['Non-HS', 'Non-Subtle HS', 'Subtle HS']

train_ds, val_ds, test_ds = load_ishate_multiclass()

for split_name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    print(f'\n{split_name}: {len(ds):,} examples')
    for task_col, names in [('label_a', TASK_A_NAMES), ('label_b', TASK_B_NAMES)]:
        task = 'A' if task_col == 'label_a' else 'B'
        counts = {n: sum(1 for x in ds if x[task_col] == i) for i, n in enumerate(names)}
        print(f'  Task {task}: {counts}')

## 4. Self-Exclusion Lookup

Builds a text → chunk_id map used during RAC training augmentation.

In [ ]:
# Build text → chunk_id lookup from the full index
with open(INDEX_DIR / 'lookup_full.json') as f:
    full_documents = json.load(f)

text_to_chunk_id = {text: int(chunk_id) for chunk_id, text in full_documents.items()}
print(f'Full index lookup loaded: {len(text_to_chunk_id):,} entries')

## 5. Helpers

In [ ]:
from training_utils import compute_metrics, tokenize_augmented, set_seed


# tokenize: plain tokenization for baseline variants
def tokenize(hf_dataset, label_col, tokenizer, text_col='text'):
    encoded = tokenizer(
        list(hf_dataset[text_col]),
        truncation=True, padding='max_length', max_length=MAX_LENGTH,
    )
    encoded['labels'] = list(hf_dataset[label_col])
    return Dataset.from_dict(encoded)


def augment_split(hf_dataset, is_train, ret_model, ret_tokenizer,
                  ret_index, ret_documents, label_col, text_col='text'):
    records = []
    for ex in tqdm(hf_dataset, desc='augmenting'):
        chunk_id  = text_to_chunk_id.get(ex[text_col]) if is_train else None
        neighbors = retrieve_top_k_above_threshold(
            ex[text_col], THRESHOLD, ret_model, ret_tokenizer,
            ret_index, ret_documents, chunk_id=chunk_id, k=K, use_mean_pool=True,
        )
        records.append({
            'query':     ex[text_col],
            'neighbors': [t for t, _ in neighbors],
            'label':     ex[label_col],
        })
    return records


def make_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        eval_strategy='epoch',
        save_strategy='no',
        logging_strategy='epoch',
        report_to='none',
        seed=42,
    )

## 6. Training & Evaluation Loop

For each task × model × variant (baseline / RAC): fine-tune and evaluate.

In [ ]:
# Models used
MODELS = {
    'bert':     'bert-base-uncased',
    'hatebert': 'GroNLP/hateBERT',
    'roberta':  'roberta-base',
}

results = {'A': {}, 'B': {}}

# Evaluate each task × model × variant combination
for task, label_col, class_names in [
    ('A', 'label_a', TASK_A_NAMES),
    ('B', 'label_b', TASK_B_NAMES),
]:
    print(f"\n{'#'*60}\nTask {task}\n{'#'*60}")

    for model_key, hf_id in MODELS.items():
        for variant in ['baseline', 'rag']:
            label = f'{model_key} ({variant})'
            print(f"\n{'='*55}\n{label} — Task {task}\n{'='*55}")

            tokenizer = AutoTokenizer.from_pretrained(hf_id)

            if variant == 'baseline':
                tok_train = tokenize(train_ds, label_col, tokenizer)
                tok_val   = tokenize(val_ds,   label_col, tokenizer)
                tok_test  = tokenize(test_ds,  label_col, tokenizer)

            else:  # rag — sbert retriever, shared across all classifier models
                print('Loading sbert retriever...')
                ret_tokenizer = AutoTokenizer.from_pretrained(SBERT_HF_ID)
                ret_model     = AutoModel.from_pretrained(SBERT_HF_ID).eval().to(device)
                ret_index     = faiss.read_index(
                    str(INDEX_DIR / 'vdb_full.faiss')
                )
                with open(INDEX_DIR / 'lookup_full.json') as f:
                    ret_docs = json.load(f)

                aug_train = augment_split(train_ds, True,  ret_model, ret_tokenizer, ret_index, ret_docs, label_col)
                aug_val   = augment_split(val_ds,   False, ret_model, ret_tokenizer, ret_index, ret_docs, label_col)
                aug_test  = augment_split(test_ds,  False, ret_model, ret_tokenizer, ret_index, ret_docs, label_col)

                del ret_model, ret_tokenizer
                if device.type == 'cuda':
                    torch.cuda.empty_cache()

                tok_train = tokenize_augmented(aug_train, tokenizer)
                tok_val   = tokenize_augmented(aug_val,   tokenizer)
                tok_test  = tokenize_augmented(aug_test,  tokenizer)

            # Seed before model init for reproducible classifier head initialization.
            # TrainingArguments(seed=42) only seeds the training loop, not from_pretrained().
            set_seed(42)
            model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=3)
            trainer = Trainer(
                model=model,
                args=make_training_args(f'./tmp_ckpt/{task}/{label}'),
                train_dataset=tok_train,
                eval_dataset=tok_val,
                compute_metrics=compute_metrics,
            )
            trainer.train()

            preds_out   = trainer.predict(tok_test)
            preds       = np.argmax(preds_out.predictions, axis=-1)
            labels_true = list(test_ds[label_col])

            print(classification_report(
                labels_true, preds,
                target_names=class_names,
                labels=list(range(len(class_names))),
            ))
            results[task][label] = {'preds': preds.tolist(), 'labels': labels_true}

            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()

## 7. Paper Results

Published results from Table 3 of Ocampo et al. (EACL 2023) for comparison.

In [ ]:
# Format: [precision, recall, f1] per class
PAPER_RESULTS = {
    'A': {
        'DeBERTa (paper)':    {'Non-HS': [0.927, 0.899, 0.825], 'Explicit HS': [0.825, 0.880, 0.851], 'Implicit HS': [0.467, 0.419, 0.442]},
        'HateBERT + ALL (paper)':       {'Non-HS': [0.903, 0.896, 0.899], 'Explicit HS': [0.827, 0.827, 0.827], 'Implicit HS': [0.502, 0.559, 0.529]},
        'DeBERTa + RI (paper)':   {'Non-HS': [0.922, 0.894, 0.908], 'Explicit HS': [0.821, 0.878, 0.849], 'Implicit HS': [0.460, 0.398, 0.427]},
    },
    'B': {
        'DeBERTa (paper)':    {'Non-HS': [0.920, 0.893, 0.906], 'Non-Subtle HS': [0.823, 0.877, 0.849], 'Subtle HS': [0.375, 0.077, 0.128]},
        'DeBERTa + BT (paper)':       {'Non-HS': [0.920, 0.897, 0.908], 'Non-Subtle HS': [0.835, 0.876, 0.855], 'Subtle HS': [0.385, 0.256, 0.308]},
        'USE + SVM + BT (paper)':   {'Non-HS': [0.892, 0.868, 0.880], 'Non-Subtle HS': [0.789, 0.831, 0.809], 'Subtle HS': [0.739, 0.436, 0.548]},
    },
}

## 8. Results — Comparison Table

One table per task. Columns: per-class Precision / Recall / F1 (matching Table 3 layout). Paper rows shown only when at least one value is filled in.

In [ ]:
for task, class_names in [('A', TASK_A_NAMES), ('B', TASK_B_NAMES)]:
    rows = {}

    # Our models
    for label, entry in results[task].items():
        p, r, f, _ = precision_recall_fscore_support(
            entry['labels'], entry['preds'],
            labels=list(range(len(class_names))), zero_division=0,
        )
        rows[label] = {
            (cls, m): v
            for cls, pi, ri, fi in zip(class_names, p, r, f)
            for m, v in [('P', pi), ('R', ri), ('F1', fi)]
        }

    # Paper results
    for label, per_class in PAPER_RESULTS[task].items():
        if all(v is None for vals in per_class.values() for v in vals):
            continue
        rows[label] = {
            (cls, m): vals[i]
            for cls, vals in per_class.items()
            for i, m in enumerate(['P', 'R', 'F1'])
        }

    df = pd.DataFrame(rows).T
    df.columns = pd.MultiIndex.from_tuples(df.columns)
    df.index.name = 'Model'

    display(
        df.style
        .format('{:.3f}', na_rep='—')
        .set_caption(f'Task {task} — ISHate (Ocampo et al., EACL 2023) comparison')
    )